<a href="https://colab.research.google.com/github/namproong/HDI-knowledge-graph/blob/main/node_embed_by_RotatE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

เริ่มทำการ embed อีกครัง

In [ ]:
import os
import gc
import json
import pickle
import pandas as pd
import torch

!pip install pykeen
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.evaluation import RankBasedEvaluator
from google.colab import drive

INFO:pykeen.utils:Using opt_einsum


In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/HDI_project_ver2'
file_path = os.path.join(base_path, 'HDI_triples_finalV2forP.parquet')
checkpoint_dir = os.path.join(base_path, 'checkpoints')
output_dir = os.path.join(base_path, 'outputs')

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("Base path:", base_path)
print("Input file:", file_path)
print("Checkpoint dir:", checkpoint_dir)
print("Output dir:", output_dir)

Base path: /content/drive/MyDrive/HDI_project_ver2
Input file: /content/drive/MyDrive/HDI_project_ver2/HDI_triples_finalV2forP.parquet
Checkpoint dir: /content/drive/MyDrive/HDI_project_ver2/checkpoints
Output dir: /content/drive/MyDrive/HDI_project_ver2/outputs


In [ ]:
print("Loading data...")
df = pd.read_parquet(file_path)

df = df.iloc[:, :3].copy()
df.columns = ['head', 'relation', 'tail']
df = df.astype(str)

print("Data loaded")
print(df.head())

print("\nRelation summary")
relation_counts = df['relation'].value_counts().sort_values(ascending=False)
num_unique_relations = relation_counts.shape[0]

print("Number of relations (no inverse):", num_unique_relations)
print(relation_counts.head(20))

relation_summary_path = os.path.join(output_dir, 'relation_summary.csv')
relation_counts.rename_axis('relation').reset_index(name='count').to_csv(
    relation_summary_path, index=False
)

print("Saved relation summary to:", relation_summary_path)

Loading data...
Data loaded
                              head           relation      tail
0  CMP_FFMVHFPLIIYYNC-OKSSEWIBSA-N  FOUND_IN_ORGANISM  NPO31296
1  CMP_FFMVHFPLIIYYNC-OKSSEWIBSA-N  FOUND_IN_ORGANISM  NPO26333
2  CMP_ISVPPMXWQFCRSS-UHFFFAOYSA-N  FOUND_IN_ORGANISM  NPO53586
3  CMP_ISVPPMXWQFCRSS-UHFFFAOYSA-N     INTERACTS_WITH      2512
4  CMP_ISVPPMXWQFCRSS-UHFFFAOYSA-N     INTERACTS_WITH      2661

Relation summary
Number of relations (no inverse): 27
relation
FOUND_IN_ORGANISM       934853
INTERACTS_WITH          457967
HAS_SPECIES              23159
BELONGS_TO_GENUS         21151
HAS_PROTEIN_CLASS        14443
BELONGS_TO_FAMILY         8187
SUBCLASS_OF_ATC4          5579
METABOLIZED_BY            4061
NEGATIVELY_MODULATES      3854
HAS_ATC5                  3793
INHIBITS                  1853
BELONGS_TO_KINGDOM        1818
METABOLIZED_TO            1328
POSITIVELY_MODULATES       972
IS_SUBCLASS_OF_PC          904
SUBCLASS_OF_ATC3           841
INDUCES                    7

In [ ]:
tf = TriplesFactory.from_labeled_triples(
    df[['head', 'relation', 'tail']].values,
    create_inverse_triples=True,
)

print("TriplesFactory created")
print("Entities:", tf.num_entities)
print("Relations (with inverse):", tf.num_relations)
print("Triples:", tf.num_triples)

del df
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TriplesFactory created
Entities: 310694
Relations (with inverse): 54
Triples: 1486111


In [ ]:
training, testing, validation = tf.split(
    ratios=[0.8, 0.1, 0.1],
    random_state=42,
)

print("Split complete")
print("Train:", training.num_triples)
print("Test:", testing.num_triples)
print("Validation:", validation.num_triples)

INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [892741, 148611, 148612]


Split complete
Train: 1188888
Test: 148611
Validation: 148612


In [ ]:
train_entities = set(training.entity_to_id.keys())
train_relations = set(training.relation_to_id.keys())

def count_unseen(factory, train_entities, train_relations):
    triples = factory.label_triples(factory.mapped_triples)
    unseen_head = sum(h not in train_entities for h, r, t in triples)
    unseen_tail = sum(t not in train_entities for h, r, t in triples)
    unseen_rel = sum(r not in train_relations for h, r, t in triples)
    return {
        "unseen_heads": unseen_head,
        "unseen_tails": unseen_tail,
        "unseen_relations": unseen_rel,
    }

valid_unseen = count_unseen(validation, train_entities, train_relations)
test_unseen = count_unseen(testing, train_entities, train_relations)

print("Validation unseen:", valid_unseen)
print("Testing unseen:", test_unseen)

Validation unseen: {'unseen_heads': 0, 'unseen_tails': 0, 'unseen_relations': 0}
Testing unseen: {'unseen_heads': 0, 'unseen_tails': 0, 'unseen_relations': 0}


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

checkpoint_name = 'rotate_v3fix.checkpoint'

try:
    result = pipeline(
        training=training,
        testing=validation,   # ใช้ validation แทนใน pipeline สำหรับเวอร์ชันนี้
        model='RotatE',
        model_kwargs=dict(
            embedding_dim=200,
        ),
        training_kwargs=dict(
            num_epochs=100,
            batch_size=512,
            checkpoint_name=checkpoint_name,
            checkpoint_directory=checkpoint_dir,
            checkpoint_frequency=5,
            checkpoint_on_failure=True,
        ),
        evaluator=RankBasedEvaluator,
        evaluator_kwargs=dict(
            filtered=True,
        ),
        device=device,
    )
    print("Training finished")

except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("GPU memory error, reduce batch_size to 256 or 128")
    else:
        raise e

INFO:pykeen.pipeline.api:=> no training loop checkpoint file found at '/content/drive/MyDrive/HDI_project_ver2/checkpoints/rotate_v3fix.checkpoint'. Creating a new file.
INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Using device: cuda


INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.training.training_loop:=> no checkpoint found at '/content/drive/MyDrive/HDI_project_ver2/checkpoints/rotate_v3fix.checkpoint'. Creating a new file.
INFO:pykeen.triples.triples_factory:Creating inverse triples.


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

INFO:pykeen.triples.triples_factory:Creating inverse triples.


...

Training batches on cuda:0:   0%|          | 0.00/4.64k [00:00<?, ?batch/s]

INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 100.


Evaluating on cuda:0:   0%|          | 0.00/149k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 291.89s seconds


Training finished


In [ ]:
print("Exporting embeddings")

entity_embeddings = result.model.entity_representations[0]().detach().cpu().numpy()

entity_labels = sorted(training.entity_to_id, key=training.entity_to_id.get)
embedding_df = pd.DataFrame(entity_embeddings, index=entity_labels)

embedding_pkl_path = os.path.join(output_dir, 'rotate3_entity_embeddings.pkl')
embedding_csv_path = os.path.join(output_dir, 'rotate3_entity_embeddings.csv')

with open(embedding_pkl_path, 'wb') as f:
    pickle.dump(embedding_df, f)

embedding_df.to_csv(embedding_csv_path)

print("Saved embeddings to:", embedding_pkl_path)
print("Saved embeddings CSV to:", embedding_csv_path)
print("Shape:", embedding_df.shape)

Exporting embeddings
Saved embeddings to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate3_entity_embeddings.pkl
Saved embeddings CSV to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate3_entity_embeddings.csv
Shape: (310694, 200)


In [ ]:
import numpy as np

In [ ]:
print("Exporting embeddings")

entity_embeddings = result.model.entity_representations[0]().detach().cpu().numpy()

print("Original shape:", entity_embeddings.shape)
print("Original dtype:", entity_embeddings.dtype)

# แปลง complex -> real + imag
entity_embeddings_real = np.concatenate(
    [entity_embeddings.real, entity_embeddings.imag],
    axis=1
)

entity_labels = sorted(training.entity_to_id, key=training.entity_to_id.get)
embedding_df = pd.DataFrame(entity_embeddings_real, index=entity_labels)

embedding_pkl_path = os.path.join(output_dir, 'rotate3con_entity_embeddings_realimag.pkl')
embedding_csv_path = os.path.join(output_dir, 'rotate3con_entity_embeddings_realimag.csv')

with open(embedding_pkl_path, 'wb') as f:
    pickle.dump(embedding_df, f)

embedding_df.to_csv(embedding_csv_path)

print("Saved embeddings to:", embedding_pkl_path)
print("Saved embeddings CSV to:", embedding_csv_path)
print("Shape:", embedding_df.shape)

Exporting embeddings
Original shape: (310694, 200)
Original dtype: complex64
Saved embeddings to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate3con_entity_embeddings_realimag.pkl
Saved embeddings CSV to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate3con_entity_embeddings_realimag.csv
Shape: (310694, 400)


In [ ]:
model_path = os.path.join(output_dir, 'rotate3_model_state_dict.pt')

torch.save(result.model.state_dict(), model_path)

print("Saved model state_dict to:", model_path)

Saved model state_dict to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate3_model_state_dict.pt


In [ ]:
split_info = {
    "train_triples": int(training.num_triples),
    "validation_triples": int(validation.num_triples),
    "test_triples": int(testing.num_triples),
    "num_entities_total": int(tf.num_entities),
    "num_relations_total_after_inverse": int(tf.num_relations),
    "device": device,
    "valid_unseen": valid_unseen,
    "test_unseen": test_unseen,
    "checkpoint_name": checkpoint_name,
    "embedding_pkl": "rotate3con_entity_embeddings_realimag.pkl",
    "embedding_csv": "rotate3con_entity_embeddings_realimag.csv",
    "model_state_dict": "rotate3_model_state_dict.pt",
}

split_info_path = os.path.join(output_dir, 'split_info.json')
with open(split_info_path, 'w', encoding='utf-8') as f:
    json.dump(split_info, f, indent=2, ensure_ascii=False)

print("Saved split info to:", split_info_path)

Saved split info to: /content/drive/MyDrive/HDI_project_ver2/outputs/split_info.json


In [ ]:
from pykeen.models import RotatE

model = RotatE(
    triples_factory=training,
    embedding_dim=200,
)

model.load_state_dict(torch.load(model_path))
model.eval()

print("Model load OK")

INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Model load OK


In [ ]:
print("Evaluating on test set")

evaluator = RankBasedEvaluator(filtered=True)

test_results = evaluator.evaluate(
    model=result.model,
    mapped_triples=testing.mapped_triples,
    additional_filter_triples=[
        training.mapped_triples,
        validation.mapped_triples,
    ],
)

test_metric_df = test_results.to_df()
print(test_metric_df.head(20))

test_metrics_csv = os.path.join(output_dir, 'rotate_test_metrics.csv')
test_metric_df.to_csv(test_metrics_csv, index=False)

print("Saved test metrics to:", test_metrics_csv)

Evaluating on test set


Evaluating on cuda:0:   0%|          | 0.00/149k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 294.33s seconds


    Side    Rank_type                              Metric         Value
0   head   optimistic           median_absolute_deviation  1.334342e+01
1   tail   optimistic           median_absolute_deviation  5.930409e+00
2   both   optimistic           median_absolute_deviation  8.895613e+00
3   head    realistic           median_absolute_deviation  1.334342e+01
4   tail    realistic           median_absolute_deviation  5.930409e+00
5   both    realistic           median_absolute_deviation  8.895614e+00
6   head  pessimistic           median_absolute_deviation  1.334342e+01
7   tail  pessimistic           median_absolute_deviation  5.930409e+00
8   both  pessimistic           median_absolute_deviation  8.895613e+00
9   head   optimistic  adjusted_geometric_mean_rank_index  9.997901e-01
10  tail   optimistic  adjusted_geometric_mean_rank_index  9.998577e-01
11  both   optimistic  adjusted_geometric_mean_rank_index  9.998270e-01
12  head    realistic  adjusted_geometric_mean_rank_index  9.997

In [ ]:
# ดูเฉพาะ both + realistic + MRR / Hits@K
wanted_metrics = ['mean_reciprocal_rank', 'hits@1', 'hits@3', 'hits@5', 'hits@10']

summary_df = test_metric_df[
    (test_metric_df['Side'] == 'both') &
    (test_metric_df['Rank_type'] == 'realistic') &
    (test_metric_df['Metric'].isin(wanted_metrics))
].copy()

print(summary_df[['Metric', 'Value']].sort_values('Metric'))

Empty DataFrame
Columns: [Metric, Value]
Index: []


In [ ]:
mrr = test_results.get_metric('both.realistic.mean_reciprocal_rank')
hits1 = test_results.get_metric('both.realistic.hits@1')
hits3 = test_results.get_metric('both.realistic.hits@3')
hits5 = test_results.get_metric('both.realistic.hits@5')
hits10 = test_results.get_metric('both.realistic.hits@10')

print("MRR   :", mrr)
print("Hits@1:", hits1)
print("Hits@3:", hits3)
print("Hits@5:", hits5)
print("Hits@10:", hits10)

MRR   : 0.43363451957702637
Hits@1: 0.38459804455928565
Hits@3: 0.4492736069335379
Hits@5: 0.4817207339968105
Hits@10: 0.5287663766477583


In [ ]:
summary_csv = os.path.join(output_dir, 'rotate_test_metrics_summary.csv')
summary_df.to_csv(summary_csv, index=False)
print("Saved summary metrics to:", summary_csv)

Saved summary metrics to: /content/drive/MyDrive/HDI_project_ver2/outputs/rotate_test_metrics_summary.csv


In [ ]:
import pandas as pd
import os


# 1) DATASET TABLE

dataset_table = pd.DataFrame({
    "Item": [
        "Total entities",
        "Total relations (with inverse)",
        "Train triples",
        "Validation triples",
        "Test triples"
    ],
    "Value": [
        tf.num_entities,
        tf.num_relations,
        training.num_triples,
        validation.num_triples,
        testing.num_triples
    ]
})


# 2) COVERAGE TABLE

coverage_table = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Unseen heads": [valid_unseen["unseen_heads"], test_unseen["unseen_heads"]],
    "Unseen tails": [valid_unseen["unseen_tails"], test_unseen["unseen_tails"]],
    "Unseen relations": [valid_unseen["unseen_relations"], test_unseen["unseen_relations"]],
})


# 3) MODEL SETUP TABLE

setup_table = pd.DataFrame({
    "Parameter": [
        "Model",
        "Embedding dimension",
        "Epochs",
        "Batch size",
        "Inverse triples",
        "Evaluation",
        "Split ratio",
        "Device"
    ],
    "Value": [
        "RotatE",
        200,
        100,
        512,
        True,
        "Filtered",
        "0.8 / 0.1 / 0.1",
        device
    ]
})


# 4) RESULT TABLE

results_table = pd.DataFrame({
    "Metric": ["MRR", "Hits@1", "Hits@3", "Hits@5", "Hits@10"],
    "Value": [
        test_results.get_metric('both.realistic.mean_reciprocal_rank'),
        test_results.get_metric('both.realistic.hits@1'),
        test_results.get_metric('both.realistic.hits@3'),
        test_results.get_metric('both.realistic.hits@5'),
        test_results.get_metric('both.realistic.hits@10'),
    ]
})


# SAVE ALL TABLES

dataset_table.to_csv(os.path.join(output_dir, "table_dataset.csv"), index=False)
coverage_table.to_csv(os.path.join(output_dir, "table_coverage.csv"), index=False)
setup_table.to_csv(os.path.join(output_dir, "table_setup.csv"), index=False)
results_table.to_csv(os.path.join(output_dir, "table_results.csv"), index=False)

print("Saved all tables to:", output_dir)


# DISPLAY TABLES

print("\n=== Dataset Table ===")
display(dataset_table)

print("\n=== Coverage Table ===")
display(coverage_table)

print("\n=== Model Setup Table ===")
display(setup_table)

print("\n=== Results Table ===")
display(results_table)


# TEXT VERSION (REPORT READY)

print("\n===== REPORT TEXT =====\n")

print("Dataset Summary:")
print(dataset_table.to_string(index=False))

print("\nCoverage Check:")
print(coverage_table.to_string(index=False))

print("\nModel Configuration:")
print(setup_table.to_string(index=False))

print("\nEvaluation Results:")
print(results_table.to_string(index=False))

Saved all tables to: /content/drive/MyDrive/HDI_project_ver2/outputs

=== Dataset Table ===


,Item,Value
0,Total entities,310694
1,Total relations (with inverse),54
2,Train triples,1188888
3,Validation triples,148612
4,Test triples,148611



=== Coverage Table ===


,Dataset,Unseen heads,Unseen tails,Unseen relations
0,Validation,0,0,0
1,Test,0,0,0



=== Model Setup Table ===


,Parameter,Value
0,Model,RotatE
1,Embedding dimension,200
2,Epochs,100
3,Batch size,512
4,Inverse triples,True
5,Evaluation,Filtered
6,Split ratio,0.8 / 0.1 / 0.1
7,Device,cuda



=== Results Table ===


,Metric,Value
0,MRR,0.433635
1,Hits@1,0.384598
2,Hits@3,0.449274
3,Hits@5,0.481721
4,Hits@10,0.528766



===== REPORT TEXT =====

Dataset Summary:
                          Item   Value
                Total entities  310694
Total relations (with inverse)      54
                 Train triples 1188888
            Validation triples  148612
                  Test triples  148611

Coverage Check:
   Dataset  Unseen heads  Unseen tails  Unseen relations
Validation             0             0                 0
      Test             0             0                 0

Model Configuration:
          Parameter           Value
              Model          RotatE
Embedding dimension             200
             Epochs             100
         Batch size             512
    Inverse triples            True
         Evaluation        Filtered
        Split ratio 0.8 / 0.1 / 0.1
             Device            cuda

Evaluation Results:
 Metric    Value
    MRR 0.433635
 Hits@1 0.384598
 Hits@3 0.449274
 Hits@5 0.481721
Hits@10 0.528766
